In [ ]:
# ===============================
# Cell 0 — Imports & bootstrap
# ===============================
import itertools
import math
import random
from pathlib import Path
from typing import Dict, Any, List, Tuple

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# --- Projeto ---
from common.config_wells import get_data_sources
from data.data_loading import DataSource

# Seq2* (janelas + utilitários)
try:
    from common.seq_preprocessing import prepare_data_seq_volve, reconstruct_true_series
except Exception:
    from common.seq_preprocessing import prepare_data_seq as prepare_data_seq_volve
    from common.seq_preprocessing import reconstruct_true_series

# ARPS (core já refatorado)
from forecast_pipeline.arps_canonical import fit_arps_canonical, forecast_canonical_from_train, ArpsParams

# Avaliação padronizada
try:
    from forecast_pipeline.jobs import evaluate_job
except Exception:
    from evaluate_job import evaluate_job

# Reconstrução série física (ancora do cumulativo)
from forecast_pipeline.analytics import _reconstruct_train_series_phys

# Logging bonito (opcional)
from forecast_pipeline.logging_utils import get_logger, phase
log = get_logger("nb.arps.grid")

np.set_printoptions(suppress=True, linewidth=160)
pd.set_option("display.max_colwidth", 120)


In [ ]:
# ===========================================
# Cell 1 — Painel de Controle (dataset/well)
# ===========================================
DATASET   = "VOLVE"            # Ex.: "VOLVE" ou "UNISIM_IV"
WELL_NAME = "15/9-F-14"        # Ajuste conforme seu data lake

# DATASET   = "UNISIM_IV"            # Ex.: "VOLVE" ou "UNISIM_IV"
# WELL_NAME = "P11"        # Ajuste conforme seu data lake

TARGET_FALLBACK = None

# Janelas para reconstrução contínua (não são hiperparâmetros de ARPS)
LAG_WINDOW = 100
HORIZON    = 300
TEST_SIZE  = 0.5
VAL_SIZE   = 0.15

# Execução
N_SAMPLES = 100            # nº de combinações aleatórias
SEED = 42                 # reprodutibilidade
PLOT = False              # deixe False para rodadas rápidas (sem poluir com gráficos)

In [ ]:
# ===========================================
# Cell 2 — Funções utilitárias (carregar/prepare)
# ===========================================
def load_single_well(dataset: str, well: str, *, target_fallback: str | None = None):
    """Resolve data source do dataset/well e carrega um DF canônico ordenado por tempo."""
    ds_list = get_data_sources()
    ds = next((d for d in ds_list if d["name"].lower() == dataset.lower()), None)
    assert ds is not None, f"Dataset '{dataset}' não encontrado no config."

    ds_one = {**ds, "wells": [well]}
    loader = DataSource(ds_one).get_loader()

    loaded = loader.load()
    df = loaded[well] if isinstance(loaded, dict) else loaded

    target_col = target_fallback or ds_one.get("target_column") or ds_one.get("load_params", {}).get("serie_name")
    assert target_col in df.columns, f"Target '{target_col}' não está no DF. Colunas: {list(df.columns)[:15]}..."

    df = df.sort_values("Tempo_Inicio_Prod").reset_index(drop=True)
    return ds_one, df, target_col


def prepare_seq2_windows(df: pd.DataFrame, target_col: str,
                         lag_window: int, horizon: int, test_size: float, val_size: float):
    """
    Prepara janelas Seq2* para reconstrução contínua e scalers.
    (ARPS não usa lag/horizon como hiperparâmetros; aqui é apenas preparação de dados.)
    """
    (X_train_s, X_val_s, X_test_s,
     y_train_s, y_val_s, y_test_s,
     scaler_X, scaler_target, y_train_original) = prepare_data_seq_volve(
        df=df,
        target_col=target_col,
        input_length=lag_window,
        output_length=horizon,
        test_size=test_size,
        val_size=val_size,
        data_aug_params=None,
    )

    # Série contínua (escalada)
    y_val_scaled_1d  = reconstruct_true_series(y_val_s).astype(float)
    y_test_scaled_1d = reconstruct_true_series(y_test_s).astype(float)

    # Série física de treino
    train_series_phys = _reconstruct_train_series_phys(y_train_original, scaler_target)

    return {
        "X_train": X_train_s,
        "X_val": X_val_s,
        "X_test": X_test_s,
        "y_train_win": y_train_s,
        "y_val_win": y_val_s,
        "y_test_win": y_test_s,
        "y_val_scaled_1d": y_val_scaled_1d,
        "y_test_scaled_1d": y_test_scaled_1d,
        "y_train_original": y_train_original,
        "train_series_phys": train_series_phys,
        "scaler_X": scaler_X,
        "scaler_target": scaler_target,
    }


In [ ]:
# ===========================================
# Cell 3 — Grid de ARPS e amostragem aleatória
# ===========================================
def build_arps_grid() -> Dict[str, List[Any]]:
    """
    Define um grid simples para um "shake-down" do ARPS.
    Mantém tudo discreto para produto cartesiano.
    """
    return dict(
        variant=["hyperbolic", "harmonic", "exponential"],
        weighting=["none", "1_over_q2", "time_decay"],
        loss=["wls", "huber"],                # robustez leve; quantile/cauchy podem ser ativados depois
        loss_delta=[0.5, 1.0, 2.0],           # escala para Huber/Cauchy
        solver=["grid", "lbfgs"],             # lbfgs cai para grid se SciPy não estiver presente
        burn_in_fraction=[0.0, 0.05, 0.10],   # descarta fração inicial do treino
        piecewise=[False, True],              # 1 change-point via BIC (se True)
        piecewise_min_delta_bic=[2.0, 5.0],   # quão melhor o BIC precisa ser
        # b_grid → usar default interno; pode expor futuramente
    )


def product_dict(grid: Dict[str, List[Any]]) -> List[Dict[str, Any]]:
    """Produto cartesiano do grid → lista de dicionários (uma config por item)."""
    keys = list(grid.keys())
    vals = [grid[k] for k in keys]
    combos = []
    for tup in itertools.product(*vals):
        combos.append({k: v for k, v in zip(keys, tup)})
    return combos


def sample_configs(grid: Dict[str, List[Any]], n: int, seed: int = 42) -> List[Dict[str, Any]]:
    """Amostra aleatoriamente n configurações (sem reposição)."""
    all_cfgs = product_dict(grid)
    if n >= len(all_cfgs):
        return all_cfgs
    rnd = random.Random(seed)
    idxs = rnd.sample(range(len(all_cfgs)), n)
    return [all_cfgs[i] for i in idxs]


In [ ]:
# =============================================================
# Cell 4 — Execução de um único experimento ARPS (função pura)
# =============================================================
def run_single_arps_experiment(
    ds_one: Dict[str, Any],
    well: str,
    prep: Dict[str, Any],
    arps_cfg: Dict[str, Any],
    *,
    plot: bool = False,
) -> Dict[str, Any]:
    """
    1) Ajusta ARPS no domínio físico (treino contínuo).
    2) Projeta val/test contínuos; volta ao domínio escalado.
    3) Avalia via evaluate_job (contrato comum).
    4) Devolve métricas de sMAPE (agg/cum) e metadados da configuração.
    """
    scaler_target = prep["scaler_target"]
    y_train_original = prep["y_train_original"]
    y_val_scaled_1d  = prep["y_val_scaled_1d"].reshape(-1)
    y_test_scaled_1d = prep["y_test_scaled_1d"].reshape(-1)

    train_series_phys = prep["train_series_phys"]
    L_train = int(train_series_phys.size)
    L_val   = int(y_val_scaled_1d.size)
    L_test  = int(y_test_scaled_1d.size)

    # --- Fit ---
    with phase(log, "ARPS.fit", well=well, variant=arps_cfg.get("variant")):
        theta: ArpsParams = fit_arps_canonical(
            q_train_phys=train_series_phys,
            variant=arps_cfg["variant"],
            weighting=arps_cfg["weighting"],
            loss=arps_cfg["loss"],
            loss_delta=arps_cfg["loss_delta"],
            burn_in_fraction=arps_cfg["burn_in_fraction"],
            solver=arps_cfg["solver"],
            piecewise=bool(arps_cfg["piecewise"]),
            piecewise_min_delta_bic=float(arps_cfg["piecewise_min_delta_bic"]),
        )

    # --- Forecast (físico) ---
    with phase(log, "ARPS.forecast"):
        yval_phys, ytest_phys = forecast_canonical_from_train(theta, L_train, L_val, L_test)

        # volta ao domínio escalado (contrato point-forecast)
        y_val_pred_scaled  = scaler_target.transform(yval_phys.reshape(-1,1)).reshape(-1)
        y_test_pred_scaled = scaler_target.transform(ytest_phys.reshape(-1,1)).reshape(-1)

    # --- Avaliação padronizada ---
    # params mínimos para evaluate_job (ARPS não usa aggregation/lag/horizon como hiperparâmetros)
    params_for_eval = dict(
        architecture_name="Arps_Canonical",
        plot=bool(plot),
        lag_window=LAG_WINDOW,
        horizon=HORIZON,
        test_size=TEST_SIZE,
        val_size=VAL_SIZE,
    )
    config_for_eval = {"wells": [well]}

    with phase(log, "ARPS.evaluate"):
        (agg_test_df, cum_test_df, gm_test,
         agg_val_df,  cum_val_df,  gm_val) = evaluate_job(
            y_test_scaled=y_test_scaled_1d,
            y_test_pred=  y_test_pred_scaled,
            y_val_scaled= y_val_scaled_1d,
            y_val_pred=   y_val_pred_scaled,
            scaler_target=scaler_target,
            y_train_original=y_train_original,
            params=params_for_eval,
            config=config_for_eval,
            well=well,
            plot=bool(plot),
            ensemble_out=None,
            x_train_main_windows=prep["X_train"],   # ativa o prefixo e o plot integrado se plot=True
        )

    # --- Extrai SMAPE (agg/cum | val/test) ---
    def _smape(df: pd.DataFrame) -> float:
        try:
            return float(df.loc[:, "SMAPE"].iloc[0])
        except Exception:
            return float("nan")

    smape_val_agg  = _smape(agg_val_df)
    smape_val_cum  = _smape(cum_val_df)
    smape_test_agg = _smape(agg_test_df)
    smape_test_cum = _smape(cum_test_df)

    return dict(
        # hiperparâmetros ARPS
        **arps_cfg,
        # parâmetros ajustados (úteis para sanity check)
        fitted_variant=theta.variant,
        qi=theta.qi, D=theta.D, b=theta.b,
        piecewise_flag=getattr(theta, "piecewise", False),
        cp_index=getattr(theta, "cp_index", None),
        # métricas
        val_smape_agg=smape_val_agg,
        val_smape_cum=smape_val_cum,
        test_smape_agg=smape_test_agg,
        test_smape_cum=smape_test_cum,
    )


In [ ]:
# ==========================================================
# Cell 5 — Orquestração: amostra N configs e roda a bateria
# ==========================================================
with phase(log, "resolve/load", dataset=DATASET, well=WELL_NAME):
    ds_one, df, target_col = load_single_well(DATASET, WELL_NAME, target_fallback=TARGET_FALLBACK)

with phase(log, "prepare_seq2_windows", L=LAG_WINDOW, H=HORIZON):
    prep = prepare_seq2_windows(
        df, target_col,
        lag_window=LAG_WINDOW, horizon=HORIZON,
        test_size=TEST_SIZE, val_size=VAL_SIZE
    )

# Constrói grid e amostra N configs
grid = build_arps_grid()
configs = sample_configs(grid, n=N_SAMPLES, seed=SEED)

results: List[Dict[str, Any]] = []
for i, cfg in enumerate(configs, 1):
    try:
        log.info(f"[{i}/{len(configs)}] Running cfg={cfg}")
        res = run_single_arps_experiment(ds_one, WELL_NAME, prep, cfg, plot=PLOT)
        res["_status"] = "ok"
    except Exception as e:
        log.exception(f"Experiment {i} failed with cfg={cfg}")
        res = {**cfg, "_status": f"error: {e}"}
    results.append(res)

results_df = pd.DataFrame(results)


In [ ]:
# ===========================================
# Cell 6 — Tabela de resultados (sMAPE)
# ===========================================
def style_results(df: pd.DataFrame):
    cols = ["val_smape_agg", "val_smape_cum", "test_smape_agg", "test_smape_cum"]
    present = [c for c in cols if c in df.columns]
    sty = (
        df.style
          .format({c: "{:.3f}" for c in present})
          .background_gradient(subset=present, cmap="Greens_r")  # verde = melhor (menor)
          .bar(subset=present, color="#ddd")
    )
    return df.style.format("{:.2f}", subset=present).highlight_min(subset=present, color='lightgreen')

# Colunas principais para inspecionar
display_cols = [
    "_status",
    "variant","weighting","loss","loss_delta","solver","burn_in_fraction","piecewise","piecewise_min_delta_bic",
    "fitted_variant","qi","D","b","piecewise_flag","cp_index",
    "val_smape_agg","val_smape_cum","test_smape_agg","test_smape_cum",
]

# Ordena por sMAPE de validação (agregado) como "proxy"
safe_df = results_df.copy()
if "val_smape_agg" in safe_df.columns:
    safe_df = safe_df.sort_values("val_smape_agg", na_position="last")
display(style_results(safe_df[display_cols].fillna("—")))


In [ ]:
# ===========================================
# Cell 7 — Resumo rápido (top-N)
# ===========================================
TOP = min(5, len(results_df))
if TOP > 0 and "val_smape_agg" in results_df.columns:
    quick = results_df.sort_values("val_smape_agg").head(TOP)
    cols = ["variant","loss","solver","burn_in_fraction","piecewise","val_smape_agg","val_smape_cum","test_smape_agg","test_smape_cum"]
    print(f"Top-{TOP} by val_smape_agg")
    display(quick[cols].reset_index(drop=True).style.format("{:.3f}", subset=[c for c in cols if "smape" in c]))


In [ ]:
# =========================
# Rerun TOP-N with plot=True (robusto a assinaturas diferentes do fit)
# =========================
import inspect
import numpy as np
import pandas as pd
from forecast_pipeline.arps_canonical import fit_arps_canonical, forecast_canonical_from_train
from forecast_pipeline.analytics import _reconstruct_train_series_phys
from forecast_pipeline.logging_utils import phase

# ---- Config rápida ----
TOP = 3  # quantos configs do results_df reexecutar com plot=True

import inspect, json, re
import numpy as np

def _func_accepts_var_kwargs(func) -> bool:
    try:
        sig = inspect.signature(func)
        return any(p.kind == inspect.Parameter.VAR_KEYWORD for p in sig.parameters.values())
    except Exception:
        return True

def _filter_kwargs_for(func, kwargs: dict) -> dict:
    if _func_accepts_var_kwargs(func):
        return kwargs
    try:
        allowed = set(inspect.signature(func).parameters.keys())
        return {k: v for k, v in kwargs.items() if k in allowed}
    except Exception:
        return kwargs

# --- NEW: robust aliasing for solver/method ---
_SOLVER_ALIASES = {
    "wls": "grid",                 # legacy label for analytic WLS → 'grid'
    "nelder": "nelder-mead",
    "nelder_mead": "nelder-mead",
    "continuous": "lbfgs",         # older name meaning "continuous optimization"
}

def _normalize_solver(raw: str) -> str:
    s = (raw or "grid").strip().lower()
    return _SOLVER_ALIASES.get(s, s)

def _parse_b_grid_repr(text: str) -> np.ndarray | None:
    if not isinstance(text, str):
        return None
    m = re.match(r"\s*linspace\(\s*([0-9eE.+-]+)\s*,\s*([0-9eE.+-]+)\s*,\s*(\d+)\s*\)\s*$", text)
    if m:
        a, b, n = float(m.group(1)), float(m.group(2)), int(m.group(3))
        return np.linspace(a, b, max(3, n))
    m = re.match(r"\s*logspace\(\s*([0-9eE.+-]+)\s*,\s*([0-9eE.+-]+)\s*,\s*(\d+)\s*\)\s*$", text)
    if m:
        a, b, n = float(m.group(1)), float(m.group(2)), int(m.group(3))
        return np.logspace(a, b, max(3, n))
    if text.strip().startswith("[") and text.strip().endswith("]"):
        try:
            arr = json.loads(text)
            return np.asarray(arr, dtype=float)
        except Exception:
            return None
    return None

def _adapt_arps_kwargs_for_fit(func, kwargs: dict) -> dict:
    """Mapeia sinônimos e filtra kwargs para a assinatura atual do fit."""
    try:
        allowed = set(inspect.signature(func).parameters.keys())
    except Exception:
        allowed = set()
    k = dict(kwargs)

    # ---- Unificar method/solver preservando a intenção original ----
    raw_solver = k.pop("solver", None)
    raw_method = k.pop("method", None)
    chosen = raw_solver if raw_solver is not None else raw_method
    if chosen is not None:
        k["solver"] = _normalize_solver(chosen)
    elif "solver" not in k:
        k["solver"] = "grid"  # default

    # allow_regime_change → piecewise
    if "allow_regime_change" in k and "piecewise" in allowed:
        k["piecewise"] = bool(k.pop("allow_regime_change"))

    # cp_search → piecewise_min_delta_bic (numérico)
    if "cp_search" in k and "piecewise_min_delta_bic" in allowed:
        val = k.pop("cp_search")
        try:
            k["piecewise_min_delta_bic"] = float(val)
        except Exception:
            pass  # mantém default quando "auto"

    # loss_scale → loss_delta
    if "loss_scale" in k and "loss_delta" in allowed:
        try:
            k["loss_delta"] = float(k.pop("loss_scale"))
        except Exception:
            k.pop("loss_scale", None)

    # b_grid: aceitar string/list → np.array; ou construir a partir de bounds se necessário
    if "b_grid" in k and k["b_grid"] is not None:
        if isinstance(k["b_grid"], str):
            parsed = _parse_b_grid_repr(k["b_grid"])
            if parsed is not None:
                k["b_grid"] = parsed
            else:
                k.pop("b_grid", None)
        elif not isinstance(k["b_grid"], np.ndarray):
            try:
                k["b_grid"] = np.asarray(k["b_grid"], dtype=float)
            except Exception:
                k.pop("b_grid", None)

    if ("b_grid" not in k or k["b_grid"] is None) and "b_grid" in allowed:
        bmin = float(k.pop("b_min", 0.1) or 0.1)
        bmax = float(k.pop("b_max", 2.0) or 2.0)
        size = int(k.pop("b_grid_size", 21) or 21)
        kind = str(k.pop("b_grid_kind", "lin")).lower()
        if kind.startswith("log"):
            bmin = max(1e-6, bmin); bmax = max(bmin * 1.001, bmax)
            k["b_grid"] = np.exp(np.linspace(np.log(bmin), np.log(bmax), max(3, size)))
        else:
            k["b_grid"] = np.linspace(bmin, bmax, max(3, size))

    # Remover auxiliares não suportados
    for drop in ("b_bounds", "D_bounds"):
        if drop in k and drop not in allowed:
            k.pop(drop, None)

    # Clamp burn-in
    if "burn_in_fraction" in k:
        k["burn_in_fraction"] = float(max(0.0, min(0.2, k["burn_in_fraction"])))

    return _filter_kwargs_for(func, k)

def _row_to_arps_kwargs(row: pd.Series) -> dict:
    """Extrai kwargs do results_df sem forçar 'grid' indevidamente."""
    def g(name, default=None):
        return row[name] if name in row.index and pd.notna(row[name]) else default

    variant   = str(g("variant",   g("arps_variant", "hyperbolic"))).lower()
    weighting = str(g("weighting", g("arps_weighting", "none"))).lower()

    # Preservar solver quando houver; cair para 'method' ou default
    raw_solver = g("solver", g("method", "grid"))
    solver = _normalize_solver(str(raw_solver).lower())

    loss = str(g("loss", "huber")).lower()
    loss_delta = g("loss_delta", g("loss_scale", 1.0))
    try: loss_delta = float(loss_delta)
    except Exception: loss_delta = 1.0

    burn_in_fraction = float(g("burn_in_fraction", 0.0) or 0.0)
    piecewise = bool(g("piecewise", g("allow_regime_change", False)))
    piecewise_min_delta_bic = g("piecewise_min_delta_bic", g("cp_search", "auto"))

    b_grid = g("b_grid", None)
    if isinstance(b_grid, str):
        parsed = _parse_b_grid_repr(b_grid)
        b_grid = parsed if parsed is not None else None
    elif b_grid is not None and not isinstance(b_grid, np.ndarray):
        try: b_grid = np.asarray(b_grid, dtype=float)
        except Exception: b_grid = None

    # Fallbacks para construir grade se necessário
    b_min = float(g("b_min", 0.1) or 0.1)
    b_max = float(g("b_max", 2.0) or 2.0)
    b_grid_size = int(g("b_grid_size", 21) or 21)
    b_grid_kind = str(g("b_grid_kind", "lin")).lower()

    return dict(
        variant=variant,
        weighting=weighting,
        solver=solver,
        loss=loss,
        loss_delta=loss_delta,
        burn_in_fraction=burn_in_fraction,
        piecewise=piecewise,
        piecewise_min_delta_bic=piecewise_min_delta_bic,
        b_grid=b_grid,
        b_min=b_min, b_max=b_max, b_grid_size=b_grid_size, b_grid_kind=b_grid_kind,
    )


# ---------------------------
# 1) Preparar dados 1x (reuso do experimento base)
# ---------------------------
if TOP > 0 and "val_smape_agg" in results_df.columns:
    with phase(log, "rerun.prepare_once", dataset=DATASET, well=WELL_NAME):
        ds_list = get_data_sources()
        ds = next((d for d in ds_list if d["name"].lower() == DATASET.lower()), None)
        assert ds is not None, f"Dataset '{DATASET}' não encontrado."
        ds_one = {**ds, "wells": [WELL_NAME]}
        loader = DataSource(ds_one).get_loader()

        loaded = loader.load()
        df_local = loaded[WELL_NAME] if isinstance(loaded, dict) else loaded
        TARGET_COL = (
            TARGET_FALLBACK
            or ds_one.get("target_column")
            or ds_one.get("load_params", {}).get("serie_name")
        )
        assert TARGET_COL in df_local.columns, f"Target '{TARGET_COL}' não está no DF."
        df_local = df_local.sort_values("Tempo_Inicio_Prod").reset_index(drop=True)

        (X_train_s_, X_val_s_, X_test_s_,
         y_train_s_, y_val_s_, y_test_s_,
         scaler_X_, scaler_target_, y_train_original_) = prepare_data_seq_volve(
            df=df_local,
            target_col=TARGET_COL,
            input_length=LAG_WINDOW,
            output_length=HORIZON,
            test_size=TEST_SIZE,
            val_size=VAL_SIZE,
            data_aug_params=None,
        )

        y_val_scaled_1d_  = reconstruct_true_series(y_val_s_).astype(float)
        y_test_scaled_1d_ = reconstruct_true_series(y_test_s_).astype(float)
        train_series_phys_ = _reconstruct_train_series_phys(y_train_original_, scaler_target_)
        L_train_, L_val_, L_test_ = int(train_series_phys_.size), int(y_val_scaled_1d_.size), int(y_test_scaled_1d_.size)

    # ---------------------------
    # 2) Reexecutar TOP-N com plot=True
    # ---------------------------
    top_df = results_df.nsmallest(min(TOP, len(results_df)), "val_smape_agg").reset_index(drop=True)
    vis_rows = []

    for rank, row in top_df.iterrows():
        arps_kwargs_raw = _row_to_arps_kwargs(row)
        arps_kwargs = _adapt_arps_kwargs_for_fit(fit_arps_canonical, arps_kwargs_raw)

        print("\n" + "="*80)
        print(f"[{rank+1}/{len(top_df)}] Re-running with plot=True")
        print("KWARGS (adapted):", {k: (v if k != "b_grid" else f"array(len={len(v)})") for k, v in arps_kwargs.items()})

        with phase(log, "rerun.fit_and_plot", rank=rank+1):
            theta = fit_arps_canonical(q_train_phys=train_series_phys_, **arps_kwargs)

            yval_phys, ytest_phys = forecast_canonical_from_train(theta, L_train_, L_val_, L_test_)
            y_val_pred_scaled  = scaler_target_.transform(yval_phys.reshape(-1,1)).reshape(-1)
            y_test_pred_scaled = scaler_target_.transform(ytest_phys.reshape(-1,1)).reshape(-1)

            params_run = dict(
                architecture_name="Arps_Canonical",
                plot=True,
                # estes campos não afetam o ARPS, mas mantêm rótulos/contratos do pipeline
                lag_window=LAG_WINDOW,
                horizon=HORIZON,
                test_size=TEST_SIZE,
                val_size=VAL_SIZE,
                # guardamos meta-infos úteis
                **{k: v for k, v in arps_kwargs_raw.items() if k not in {"b_grid"}}
            )
            config_run = {"wells": [WELL_NAME]}

            (agg_test_df, cum_test_df, gm_test,
             agg_val_df,  cum_val_df,  gm_val) = evaluate_job(
                y_test_scaled=y_test_scaled_1d_,
                y_test_pred=  y_test_pred_scaled,
                y_val_scaled= y_val_scaled_1d_,
                y_val_pred=   y_val_pred_scaled,
                scaler_target=scaler_target_,
                y_train_original=y_train_original_,
                params=params_run,
                config=config_run,
                well=WELL_NAME,
                plot=True,
                ensemble_out=None,
                x_train_main_windows=X_train_s_,
            )

            vis_rows.append({
                "rank": rank+1,
                "variant": arps_kwargs_raw.get("variant"),
                "method/solver": arps_kwargs_raw.get("method", arps_kwargs_raw.get("solver", "grid")),
                "loss": arps_kwargs_raw.get("loss"),
                "burn_in": arps_kwargs_raw.get("burn_in_fraction"),
                "regime_change": arps_kwargs_raw.get("allow_regime_change", arps_kwargs_raw.get("piecewise", False)),
                "val_smape_agg_rerun": gm_val.get("SMAPE", np.nan),
                "test_smape_agg_rerun": gm_test.get("SMAPE", np.nan),
                "orig_val_smape_agg": row.get("val_smape_agg", np.nan),
                "orig_test_smape_agg": row.get("test_smape_agg", np.nan),
            })

    vis_table = pd.DataFrame(vis_rows).sort_values("rank")
    display_cols = [
        "rank", "variant", "method/solver", "loss", "burn_in", "regime_change",
        "orig_val_smape_agg", "val_smape_agg_rerun",
        "orig_test_smape_agg", "test_smape_agg_rerun",
    ]
    print("\n=== 🔎 Rerun summary (SMAPE) ===")
    display(vis_table[display_cols])

else:
    print("Nada a reexecutar: defina TOP>0 e garanta que 'val_smape_agg' exista em results_df.")
